# Task 4: Game Title Detection

- Type: 5-class image classification of 230×120 gameplay frames. Metric: macro F1.
  Labels: 0 = 7-11 Knights, 1 = FiveN, 2 = Free Fried, 3 = PUBE, 4 = Roblock.
- Pretrained weights are allowed for task 4 only (all backbones come from `timm`).
- Final model
  1. 13 pretrained backbones ("families"), each fine-tuned end-to-end on all training images with
     `src/ft.py`. There are 43 training runs in total (2 to 6 seeds per family).
  2. Blend: the seeds are averaged inside each family, then the 13 families are averaged with equal weight.
  3. Balanced-prior decision rule: argmax p / π^2.0, where π is the *training* class share. No test information
     is used; the strength 2.0 was chosen on the public LB.

In [ ]:
TRAIN = True  # True = retrain every model from the pretrained weights (GPU); False = use our saved probabilities

In [2]:
import os
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "common").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from common import paths, utils

TASK = 4
DATA = paths.task_data(TASK)
SRC = ROOT / "task4" / "src"
OUT = paths.outputs_dir(TASK)
PROBS = OUT / ("retrain" if TRAIN else "saved_probs")  # where the per-run test probabilities are read from
LABELS = ["7-11k", "fiveN", "freefried", "pube", "roblock"]

## 1. Load Data

In [3]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
test_ids = test["file_name"].str.removesuffix(".jpg")
counts = train["label"].value_counts().sort_index()
TRAIN_SHARE = (counts / counts.sum()).to_numpy()  # π: training class shares, used by the decision rule
print(f"train {len(train):,} images | test {len(test):,} images")
pd.DataFrame({"game": LABELS, "train images": counts.values, "share": TRAIN_SHARE.round(3)})

train 31,546 images | test 25,889 images


,game,train images,share
0,7-11k,12429,0.394
1,fiveN,4407,0.140
2,freefried,5596,0.177
3,pube,5926,0.188
4,roblock,3188,0.101


## 2. Training recipe
- Input: the full frame resized (224×448 by default; 126×238 / 144×272 for the "native-size" families, close to
  the original 120×230), no padding.
- Loss: cross-entropy with label smoothing 0.1 and class weights (max count / count)^0.5.
- Optimiser: AdamW (weight decay 0.05), OneCycle schedule (10% warm-up, cosine decay), layer-wise LR decay for
  ViTs, mixed precision, gradient clipping 2.0, gradient checkpointing. The model of the last epoch is used.
- Augmentation on the GPU, safe for on-screen text (no horizontal flip): random crop 75–100% of the area,
  aspect 1.7–2.2; rotation ±8° (p = 0.3); colour jitter (p = 0.8); colour inversion (p = 0.05); Gaussian noise (p = 0.3).
  Some runs add HUD dropout (--hud): an edge band or corner, where game HUDs sit, is hidden so the model must also
  use the scene itself.
- Optional per run: LP-FT (--lp-epochs 2: train the new head on the frozen backbone first), and for the two giant
  models --freeze-blocks (only the last 12 transformer blocks are trained).
- Prediction: softmax averaged over 3 zoom levels (1.0, 1.08, 1.15).

Model choice: whole scenes (CLIP k-means clusters) were held out for validation, because random folds leak near-duplicate
frames. Families after the first two were added one at a time and kept only if the public LB improved.

In [ ]:
COMMON = "--cv full --grad-ckpt"
CNXT_S = "--backbone convnext_small.fb_in22k_ft_in1k --drop-path 0.2"
CNXT_V2B = "--backbone convnextv2_base.fcmae_ft_in22k_in1k --batch 32 --lr 1e-4 --drop-path 0.2"
EVA_B = "--backbone eva02_base_patch14_448.mim_in22k_ft_in22k_in1k --batch 32 --lr 5e-5 --layer-decay 0.75"
EVA_L = "--backbone eva02_large_patch14_448.mim_in22k_ft_in22k_in1k --batch 16 --lr 3e-5 --layer-decay 0.85 --drop-path 0.2"
EVA_L_M38M = "--backbone eva02_large_patch14_448.mim_m38m_ft_in22k_in1k --batch 16 --lr 3e-5 --layer-decay 0.85 --drop-path 0.2"
DINO3_B = "--backbone vit_base_patch16_dinov3.lvd1689m --batch 32 --layer-decay 0.75"
DINO2_L = "--backbone vit_large_patch14_reg4_dinov2.lvd142m --batch 16 --lr 1e-4 --layer-decay 0.85 --drop-path 0.2"
DINO3_L = "--backbone vit_large_patch16_dinov3.lvd1689m --batch 16 --lr 1e-4 --layer-decay 0.85 --drop-path 0.2"
NATIVE = "--batch 64 --drop-path 0.2 --hud 0.7"  # near-native input size, long schedule, HUD dropout
DINO3_L_N = f"--backbone vit_large_patch16_dinov3.lvd1689m {NATIVE} --h 144 --w 272 --epochs 18 --lr 8e-5 --layer-decay 0.9"
DINO2_L_N = f"--backbone vit_large_patch14_reg4_dinov2.lvd142m {NATIVE} --h 126 --w 238 --epochs 15 --lr 3e-5 --layer-decay 0.85"
EVA_L_N = (f"--backbone eva02_large_patch14_448.mim_in22k_ft_in22k_in1k {NATIVE} --h 126 --w 238 --epochs 15 --lr 5e-5 "
           "--layer-decay 0.85 --lp-epochs 2")
DINO2_G_N = (f"--backbone vit_giant_patch14_reg4_dinov2.lvd142m {NATIVE} --h 126 --w 238 --epochs 12 --lr 3e-5 "
             "--layer-decay 0.9 --freeze-blocks 28")
DINO3_HP_N = (f"--backbone vit_huge_plus_patch16_dinov3.lvd1689m {NATIVE} --h 144 --w 272 --epochs 12 --lr 6e-5 "
              "--layer-decay 0.9 --freeze-blocks 20")

def runs(prefix, args, seeds):
    return [(f"{prefix}_s{s}", f"{args} --seed {s}") for s in seeds]

FAMILIES = {
    "ConvNeXt-S (IN-22k)": runs("f_small", CNXT_S, [42, 7, 123, 2026]) + runs("f_small_hud", f"{CNXT_S} --batch 24 --hud 0.5", [42]),
    "ConvNeXt-V2-B (FCMAE)": runs("f_v2base", CNXT_V2B, [42, 7, 123]),
    "EVA-02-B (MIM IN-22k)": runs("f_eva02b", EVA_B, [42, 7, 123, 2026]) + runs("f_eva02b_lpft", f"{EVA_B} --lp-epochs 2", [42, 7]),
    "EVA-02-L (MIM IN-22k)": runs("f_eva02l", EVA_L, [42, 7, 123, 2026]) + runs("f_eva02l_lpft", f"{EVA_L} --lp-epochs 2", [42]),
    "EVA-02-L (MIM M38M)": runs("f_eva02lm38", EVA_L_M38M, [42, 7]),
    "DINOv3-B": runs("f_dinov3b_lr2e4", DINO3_B, [42, 7]) + runs("f_dinov3b_hud", f"{DINO3_B} --hud 0.5", [42]),
    "DINOv2-L": runs("f_dinov2l", DINO2_L, [42, 7]),
    "DINOv3-L": runs("f_dinov3l", DINO3_L, [42, 7]),
    "DINOv3-L native 144×272": runs("f_dinov3l_n144", DINO3_L_N, [42, 7, 123]),
    "DINOv2-L native 126×238": runs("f_dinov2l_n126", DINO2_L_N, [42, 7, 123]),
    "EVA-02-L native 126×238": runs("f_eva02l_n126", EVA_L_N, [42, 7, 123]),
    "DINOv2-g native 126×238": runs("f_dinov2g_n126", DINO2_G_N, [42, 7, 123]),
    "DINOv3-H+ native 144×272": runs("f_dinov3hp_n144", DINO3_HP_N, [42, 7, 123]),
}
print(f"{len(FAMILIES)} families, {sum(len(v) for v in FAMILIES.values())} training runs")
pd.DataFrame([(fam, len(r), r[0][1].split()[1]) for fam, r in FAMILIES.items()], columns=["family", "runs", "timm backbone"])

13 families, 43 training runs


,family,runs,timm backbone
0,ConvNeXt-S (IN-22k),5,convnext_small.fb_in22k_ft_in1k
1,ConvNeXt-V2-B (FCMAE),3,convnextv2_base.fcmae_ft_in22k_in1k
2,EVA-02-B (MIM IN-22k),6,eva02_base_patch14_448.mim_in22k_ft_in22k_in1k
3,EVA-02-L (MIM IN-22k),5,eva02_large_patch14_448.mim_in22k_ft_in22k_in1k
4,EVA-02-L (MIM M38M),2,eva02_large_patch14_448.mim_m38m_ft_in22k_in1k
5,DINOv3-B,3,vit_base_patch16_dinov3.lvd1689m
6,DINOv2-L,2,vit_large_patch14_reg4_dinov2.lvd142m
7,DINOv3-L,2,vit_large_patch16_dinov3.lvd1689m
8,DINOv3-L native 144×272,3,vit_large_patch16_dinov3.lvd1689m
9,DINOv2-L native 126×238,3,vit_large_patch14_reg4_dinov2.lvd142m


## 3. Training

In [ ]:
if TRAIN:
    env = {**os.environ, "T4_OUT_DIR": str(PROBS), "T4_CACHE_DIR": str(OUT)}
    PROBS.mkdir(parents=True, exist_ok=True)
    for fam, fam_runs in FAMILIES.items():
        for name, args in fam_runs:
            if (PROBS / f"test_{name}.npy").exists():
                print(f"skip {name} (done)")
                continue
            t0 = time.time()
            cmd = [sys.executable, "ft.py", "--name", name, *COMMON.split(), *args.split()]
            print(" ".join(cmd[1:]), flush=True)
            subprocess.run(cmd, cwd=SRC, env=env, check=True)
            print(f"  {name} done in {(time.time() - t0) / 60:.0f} min", flush=True)
else:
    print("TRAIN = False: using the saved probabilities in", PROBS.relative_to(ROOT))

TRAIN = False: using the saved probabilities in task4/outputs/saved_probs


## 4. Blend and decision rule

In [ ]:
PRIOR_A = 2.0
fam_p = {fam: np.mean([np.load(PROBS / f"test_{name}.npy") for name, _ in fam_runs], 0)
         for fam, fam_runs in FAMILIES.items()}
p = np.mean(list(fam_p.values()), 0)
p_adj = p / TRAIN_SHARE ** PRIOR_A
p_adj = p_adj / p_adj.sum(1, keepdims=True)
pred = p_adj.argmax(1)

agree = pd.Series({fam: (q.argmax(1) == pred).mean() for fam, q in fam_p.items()}, name="agreement with final")
mix = pd.DataFrame({"train share": TRAIN_SHARE, "test pred, no rule": np.bincount(p.argmax(1), minlength=5) / len(p),
                    "test pred, final": np.bincount(pred, minlength=5) / len(pred)}, index=LABELS)
print(f"rows changed by the decision rule: {int((pred != p.argmax(1)).sum()):,} of {len(pred):,}")
display(agree.round(4).to_frame())
mix.round(3)

rows changed by the decision rule: 1,172 of 25,889


,agreement with final
ConvNeXt-S (IN-22k),0.8792
ConvNeXt-V2-B (FCMAE),0.8978
EVA-02-B (MIM IN-22k),0.9081
EVA-02-L (MIM IN-22k),0.9296
EVA-02-L (MIM M38M),0.9260
DINOv3-B,0.9164
DINOv2-L,0.9367
DINOv3-L,0.9394
DINOv3-L native 144×272,0.9364
DINOv2-L native 126×238,0.9343


,train share,"test pred, no rule","test pred, final"
7-11k,0.394,0.150,0.127
fiveN,0.140,0.269,0.277
freefried,0.177,0.150,0.149
pube,0.188,0.247,0.236
roblock,0.101,0.182,0.210


## 5. Write output

In [7]:
final = utils.write_final_pred(TASK, test_ids, pred)
final.head()

,id,prediction
0,sjqyfwc,1
1,s07kzz5,2
2,sr69rma,3
3,soqvor4,0
4,s2v4jw3,3
